# Ava-LLM Pre-Training Notebook
**Hybrid Mamba-2 + Transformer** language model pre-training on raw Georgian text.

Supports three architecture modes:
- **`hybrid-130m`** — 10 Mamba SSM + 2 Attention layers (recommended for T4)
- **`mamba-130m`** — Pure Mamba SSM (fastest inference)
- **`100m`** — Pure Transformer (baseline)

**Pipeline**:
1. **Pre-train** (this notebook) — raw CulturaX Georgian corpus → base weights
2. **Fine-tune** (separate) — conversational data → chat model

Features: AMP (mixed precision), gradient accumulation, cosine LR schedule, SDPA/FlashAttention, parallel scan.

In [ ]:
# Uncomment to clone fresh repo in Colab
# !rm -rf ./* ./.*
# !git clone https://github.com/Kuduxaaa/ava-llm .
# !rm -rf checkpoints

## 1. Imports

In [ ]:
import os
import torch
import numpy as np

from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from ava import AvaConfig, AvaForCausalLM
from ava.data.datasets import PretrainDataset
from ava.training import TrainingConfig, train_model
from ava.utils import collate_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()} | {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Model & Tokenizer Configuration

Choose architecture preset:
| Preset | Type | Params | VRAM (AMP) |
|--------|------|--------|------------|
| `hybrid-130m` | 10 Mamba + 2 Attn | ~103M | ~8 GB |
| `mamba-130m` | Pure Mamba | ~93M | ~6 GB |
| `100m` | Pure Transformer | ~101M | ~10 GB |

In [ ]:
# ── Architecture ──────────────────────────────────────────────
MODEL_PRESET = "hybrid-130m"  # "hybrid-130m" | "mamba-130m" | "100m"

config = AvaConfig().apply_for(MODEL_PRESET)

# ── Tokenizer ─────────────────────────────────────────────────
# Custom SentencePiece BPE tokenizer trained on CulturaX Georgian corpus
# To train it, run notebooks/train_tokenizer.ipynb first
TOKENIZER_PATH = "../data/tokenizer"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

# ── Sync config with tokenizer ────────────────────────────────
config.vocab_size = len(tokenizer)
config.pad_token_id = tokenizer.pad_token_id
config.bos_token_id = tokenizer.bos_token_id
config.eos_token_id = tokenizer.eos_token_id
config.tie_word_embeddings = True

print(f"Architecture: {config.architecture_type}")
print(f"Layers: {config.num_hidden_layers} (hidden={config.hidden_size})")
print(f"Vocab: {config.vocab_size}")

## 3. Dataset

Loads raw Georgian text from CulturaX corpus for causal language model pre-training.

> **Prerequisites**: Run `scripts/download_culturax.py` and `notebooks/train_tokenizer.ipynb` first.

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────
MAX_SEQ_LENGTH = 512
BATCH_SIZE = 4
TRAIN_RATIO = 0.9

# ── Load corpus ───────────────────────────────────────────────
CORPUS_PATH = "../data/corpus_ka.txt"

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    texts = [line.strip() for line in f if len(line.strip()) >= 50]

print(f"Loaded {len(texts):,} documents from {CORPUS_PATH}")

np.random.seed(42)
np.random.shuffle(texts)

split_idx = int(len(texts) * TRAIN_RATIO)
train_dataset = PretrainDataset(texts[:split_idx], tokenizer, max_length=MAX_SEQ_LENGTH)
val_dataset = PretrainDataset(texts[split_idx:], tokenizer, max_length=MAX_SEQ_LENGTH)

print(f"Train: {len(train_dataset):,} samples | Val: {len(val_dataset):,} samples")

assert len(train_dataset) > 0 and len(val_dataset) > 0, "Empty dataset!"

# ── DataLoaders ───────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=(device.type == "cuda"),
    num_workers=2,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    pin_memory=(device.type == "cuda"),
    num_workers=2,
)

## 4. Verify Data Pipeline

In [ ]:
batch = next(iter(train_loader))
print(f"Batch shapes:")
print(f"  input_ids:      {batch['input_ids'].shape}")
print(f"  attention_mask:  {batch['attention_mask'].shape}")
print(f"  labels:          {batch['labels'].shape}")

max_id = batch["input_ids"].max().item()
padded = (batch["labels"] == -100).sum().item()
total = batch["labels"].numel()
print(f"\nMax token ID: {max_id} (vocab: {config.vocab_size})")
print(f"Padded tokens: {padded}/{total} ({100*padded/total:.0f}%) — padding excluded from loss")

assert max_id < config.vocab_size, f"Token ID {max_id} >= vocab size!"

## 5. Create Model

In [ ]:
model = AvaForCausalLM(config)

# Resize embeddings if tokenizer has extra tokens
if config.vocab_size != model.model.embed_tokens.weight.shape[0]:
    model.model.embed_tokens = torch.nn.Embedding(config.vocab_size, config.hidden_size)
    model.lm_head = torch.nn.Linear(config.hidden_size, config.vocab_size, bias=False)
    if config.tie_word_embeddings:
        model.lm_head.weight = model.model.embed_tokens.weight

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Layer breakdown
mamba_layers = sum(1 for l in model.model.layers if getattr(l, "is_mamba", False))
attn_layers = len(model.model.layers) - mamba_layers

print(f"Parameters: {n_params:,} ({n_params/1e6:.0f}M)")
print(f"Trainable:  {n_trainable:,}")
print(f"Layers:     {mamba_layers} Mamba + {attn_layers} Attention = {len(model.model.layers)} total")
print(f"Est. VRAM:  ~{n_params * 2 * 4 / 1e9:.1f} GB (AMP, batch={BATCH_SIZE})")

## 6. Training

Training with:
- **AMP** (mixed precision) — ~2x speedup + ~2x less VRAM
- **Gradient accumulation** — larger effective batch without OOM
- **Cosine LR schedule** — linear warmup → cosine decay
- **SDPA/FlashAttention** — fused attention kernels
- **Parallel scan** — O(log L) Mamba on GPU

In [ ]:
training_config = TrainingConfig(
    num_epochs=3,
    learning_rate=5e-4,
    weight_decay=0.1,
    max_grad_norm=1.0,
    use_amp=True,
    gradient_accumulation_steps=8,   # effective batch = 4 * 8 = 32
    warmup_ratio=0.05,
    log_interval=5,
    checkpoint_dir="checkpoints",
    compile_model=False,             # set True for PyTorch 2.x speedup (slower first step)
)

try:
    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        training_config=training_config,
    )
except KeyboardInterrupt:
    print("\nTraining interrupted by user.")
except Exception as e:
    print(f"Training error: {e}")
    import traceback; traceback.print_exc()

## 7. Training Curves

In [ ]:
import matplotlib.pyplot as plt

if "history" in dir() and history:
    epochs = [h["epoch"] for h in history]
    train_losses = [h["train_loss"] for h in history]
    val_losses = [h["val_loss"] for h in history if h["val_loss"] is not None]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, train_losses, "b-o", label="Train")
    if val_losses:
        ax1.plot(epochs[:len(val_losses)], val_losses, "r-o", label="Val")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss Curve")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    import math
    train_ppl = [math.exp(min(l, 20)) for l in train_losses]
    ax2.plot(epochs, train_ppl, "b-o", label="Train PPL")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Perplexity")
    ax2.set_title("Perplexity")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No training history available.")

## 8. Save Model

In [ ]:
save_path = "ava_model_trained.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "config": config.to_dict(),
    "tokenizer_path": TOKENIZER_PATH,
}, save_path)

size_mb = os.path.getsize(save_path) / 1e6 if os.path.exists(save_path) else 0
print(f"Model saved: {save_path} ({size_mb:.0f} MB)")

## 9. Text Generation (Sanity Check)

Pre-trained მოდელი ჯერ კიდევ არ არის fine-tuned — ელოდეთ არათანმიმდევრულ, მაგრამ ქართულ ტექსტს.

In [ ]:
model.eval()

prompts = [
    "საქართველო არის",
    "ხელოვნური ინტელექტი",
]

for prompt in prompts:
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=150,
            temperature=0.7,
            top_k=50,
            top_p=0.9,
            repetition_penalty=1.2,
        )
        
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"{'='*60}")
    print(response)
    print()